In [1]:
from swemnics.problems import TidalProblem
from swemnics import solvers as Solvers
from mpi4py import MPI
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dolfinx import fem as fe
from tqdm import tqdm

import pandas as pd
from scipy.optimize import minimize
import seaborn as sns 
from typing import Callable, Dict, List, Tuple, Any

from fourd_var import run_assimilation
from dca_utils import*
from plotting import plot_simulation_results, create_comparison_figure

sns.set_palette("bright")
plt.style.use("mystyle1.mplstyle")

In [2]:
comm = MPI.COMM_WORLD
rank = comm.Get_rank()

problem_params = {
    'nx': 20,
    'ny': 5,
    'dt': 3600,
    't': 0,
    't_final': 7*24*60*60, # 7 days 
    'num_steps': int(np.ceil((7*24*60*60)/3600)),
    'num_windows': 4,
    'fric_law': 'mannings',    #friction law either quadratic or linear
    'sol_var': 'h'             #solution variable either h or hu
}

solver_params = {"rtol": 1e-5,
          "atol": 1e-6,
          "max_it":10,
          "relaxation_parameter":1.0,
          "ksp_type": "gmres",
          "pc_type": "ilu",
          "ksp_ErrorIfNotConverged": False,
          }#,"pc_factor_mat_solver_type":"mumps"}


In [3]:
assert problem_params['num_steps'] == int(np.ceil(problem_params['t_final']/problem_params['dt']))
obs_frequency = 4  # observe system every other time step
true_signal, prob, stations, state_coords = get_true_signal(problem_params,
                                                            'tidal',
                                                            solver_params, 
                                                            obs_frequency)

[Rank 0] Inside save_states – Before gathered_states
[Rank 0] Inside save_states – Before concatenate
[Rank 0] Inside save_states – gathered state size (378,)
[Rank 0] Inside save_states – After gathered_states
[Rank 0] Inside save_states – Before append
[Rank 0] Inside save_states – state size (378,)
[Rank 0] Inside save_states – After append
[Rank 0] Inside save_adjoint – Before assemble_A
[Rank 0] Inside assemble_A – Before assemble_matrix
[Rank 0] Inside assemble_A – Before assemble_matrix
[Rank 0] Inside save_adjoint – After assemble_A
[Rank 0] Inside save_adjoint – Before petsc_to_csr
[Rank 0] Inside save_adjoint – After petsc_to_csr
[Rank 0] Inside save_adjoint – After saved_adjoints
[Rank 0] Inside save_states – Before gathered_states
[Rank 0] Inside save_states – Before concatenate
[Rank 0] Inside save_states – gathered state size (378,)
[Rank 0] Inside save_states – After gathered_states
[Rank 0] Inside save_states – Before append
[Rank 0] Inside save_states – state size (378

In [5]:
true_signal.u.x.array.shape

(378,)

In [4]:
print(f"Total Dof =  H_dof x u_dof x v_dof: {true_signal.u.x.array.shape}")
print(f"Observation Spatial Frequency: {obs_frequency}")
print(f"Number of Stations x (H,u,v): {stations.shape}")
print(f"N Time Steps, N Stations, N Variables: {true_signal.vals.shape}")
print(f"Number of Adjoints Saved: {len(true_signal.saved_adjoints)}")
print(f"Adjoint Size: {true_signal.saved_adjoints[0].shape}")
print(f"Number of states Saved: {len(true_signal.saved_states)}")
print(f"State Size: {true_signal.saved_states[0].shape}")

Total Dof =  H_dof x u_dof x v_dof: (378,)
Observation Spatial Frequency: 4
Number of Stations x (H,u,v): (32, 3)
N Time Steps, N Stations, N Variables: (169, 32, 3)
Number of Adjoints Saved: 168
Adjoint Size: (378, 378)
Number of states Saved: 168
State Size: (378,)


In [5]:
obs_std = 2.2
obs_time_freq = 4
total_steps = int((problem_params['t_final']/problem_params['dt']) + 1)
problem_params['num_steps'] = int(np.ceil((7*24*60*60)/3600)/problem_params['num_windows']) # Size of each assimilation window
obs_per_window = problem_params['num_steps'] // obs_time_freq 

print(f"Total Steps: {total_steps}\n"
      f"Total Assimilation Windows: {problem_params['num_windows']}\n"
      f"Steps per Window: {problem_params['num_steps']}\n"
      f"Obs Frequency: {obs_time_freq}\n"
      f"Total Obs: {obs_per_window * problem_params['num_windows']}\n"
      f"Number Stations: {stations.shape[0]}\n"
      f"Obs per Window: {obs_per_window}\n")


# Create synthetic observations
obs_spatial_indices = find_obs_indices(stations, state_coords)
obs_indices_per_window, obs_time_indices = setup_observation_indices(problem_params['num_steps'], obs_time_freq, total_steps)

print(f"Observation Spatial Indices: {obs_spatial_indices}\n\n"
      f"Observation Time Indices: {obs_time_indices}\n\n"
      f"Observation Time Indices per Window: {obs_indices_per_window}\n\n"
      )

hb = 10 + (stations[:, 0]*0)
y_obs = generate_observations(true_signal, obs_time_indices, obs_std)


# Generate Background,Observation, and Predicted Error Covariance Matrices
state_dim = true_signal.saved_adjoints[0].shape[0]
obs_dim = stations.shape[0]

# Observation Matrix
H = np.zeros((obs_dim, state_dim))
H[np.arange(obs_dim), obs_spatial_indices] = 1.0

# Observation Covariance 
R = np.eye(obs_dim) * (obs_std**2)


inflation_factor=3.5
B = inflation_factor*np.eye(state_dim) 

# Predicted Covariance
L = H @ B @ H.T 

# Get Inverse Covariance matrices
R_inv = np.linalg.inv(R)
B_inv = np.linalg.inv(B) 
L_inv = np.linalg.inv(L)

covs = {"B_inv": B_inv, "R_inv": R_inv, "L_inv": L_inv}

print(f"State Dimension: {state_dim}\n"
      f"Observation Dimension: {obs_dim}\n"
      f"Background Covariance Matrix Shape B: {B.shape}\n"
      f"Observation Covariance Matrix Shape R: {R.shape}\n"
      f"Predicted Error Covariance Matrix shape L: {L.shape}\n"
      f"Observation Matrix Shape H: {H.shape}\n")


Total Steps: 169
Total Assimilation Windows: 4
Steps per Window: 42
Obs Frequency: 4
Total Obs: 40
Number Stations: 32
Obs per Window: 10

Observation Spatial Indices: [  0   4   8  12  16  20  24  28  32  36  40  44  48  52  56  60  64  68
  72  76  80  84  88  92  96 100 104 108 112 116 120 124]

Observation Time Indices: [  0   4   8  12  16  20  24  28  32  36  40  44  48  52  56  60  64  68
  72  76  80  84  88  92  96 100 104 108 112 116 120 124 128 132 136 140
 144 148 152 156 160 164]

Observation Time Indices per Window: [ 0  4  8 12 16 20 24 28 32 36 40]


State Dimension: 378
Observation Dimension: 32
Background Covariance Matrix Shape B: (378, 378)
Observation Covariance Matrix Shape R: (32, 32)
Predicted Error Covariance Matrix shape L: (32, 32)
Observation Matrix Shape H: (32, 378)



In [6]:
bayes_analysis = run_assimilation(
                                  problem_params,
                                  solver_params,
                                  stations,
                                  y_obs,
                                  obs_per_window,
                                  obs_spatial_indices,
                                  obs_time_indices,
                                  H,
                                  covs,
                                  hb,
                                'tidal',
                                cost_function_type='bayes'
                             )


Processing windows:   0%|          | 0/4 [00:00<?, ?window/s]

Solver Time 1: 0
Iteration 1: Cost = 0.000000
Iteration 2: Cost = 1.000000

Optimization completed:
  Success: False
  Status: -6
  Message: Optimization completed successfully
  Final cost: 1.233488e+03
  Iterations: 2
  Function evaluations: 32
  Gradient evaluations: 32
  Gradient norm at solution: 3.192840e-03

------------------------------------------------------------

State comparison (subsampled):
  Initial state (every 100th entry):   [10.  0.  0. 10.]

  Optimized state (every 100th entry): [10.  0.  0. 10.]

Solver Time 2: 0


Processing windows:  25%|██▌       | 1/4 [00:43<02:09, 43.22s/window]

/////////////////////////////////////// Window 1 Completed ////////////////////////////////////////////////// 


Solver Time 1: 151200
Iteration 1: Cost = 0.000000
Iteration 2: Cost = 1.000000

Optimization completed:
  Success: False
  Status: -6
  Message: Optimization completed successfully
  Final cost: 1.260528e+03
  Iterations: 2
  Function evaluations: 32
  Gradient evaluations: 32
  Gradient norm at solution: 4.126225e-03

------------------------------------------------------------

State comparison (subsampled):
  Initial state (every 100th entry):   [ 9.88850667e+00 -5.69511811e-03 -3.07018803e-07  9.88937361e+00]

  Optimized state (every 100th entry): [ 9.88850667e+00 -5.69511811e-03 -3.07018803e-07  9.88937361e+00]

Solver Time 2: 151200


Processing windows:  50%|█████     | 2/4 [01:29<01:30, 45.19s/window]

/////////////////////////////////////// Window 2 Completed ////////////////////////////////////////////////// 


Solver Time 1: 302400
Iteration 1: Cost = 0.000000
Iteration 2: Cost = 1.000000

Optimization completed:
  Success: False
  Status: -6
  Message: Optimization completed successfully
  Final cost: 1.335572e+03
  Iterations: 2
  Function evaluations: 32
  Gradient evaluations: 32
  Gradient norm at solution: 5.517292e-03

------------------------------------------------------------

State comparison (subsampled):
  Initial state (every 100th entry):   [1.00122626e+01 8.02623215e-03 6.50201523e-07 1.00122134e+01]

  Optimized state (every 100th entry): [1.00122626e+01 8.02623215e-03 6.50201523e-07 1.00122134e+01]

Solver Time 2: 302400


Processing windows:  75%|███████▌  | 3/4 [02:12<00:43, 43.88s/window]

/////////////////////////////////////// Window 3 Completed ////////////////////////////////////////////////// 


Solver Time 1: 453600
Iteration 1: Cost = 0.000000
Iteration 2: Cost = 1.000000

Optimization completed:
  Success: False
  Status: -6
  Message: Optimization completed successfully
  Final cost: 1.315980e+03
  Iterations: 2
  Function evaluations: 32
  Gradient evaluations: 32
  Gradient norm at solution: 7.422892e-03

------------------------------------------------------------

State comparison (subsampled):
  Initial state (every 100th entry):   [ 1.00935602e+01 -6.12777387e-03 -4.76003987e-08  1.00926917e+01]

  Optimized state (every 100th entry): [ 1.00935602e+01 -6.12777387e-03 -4.76003987e-08  1.00926917e+01]

Solver Time 2: 453600


Processing windows: 100%|██████████| 4/4 [02:52<00:00, 43.12s/window]

/////////////////////////////////////// Window 4 Completed ////////////////////////////////////////////////// 




In [7]:

bayes_height_misfit_rmse = np.sqrt(np.mean((true_signal.vals[:,:,0] - bayes_analysis[:,:,0]) ** 2))
bayes_total_misfit_rmse = np.sqrt(np.mean((true_signal.vals - bayes_analysis) ** 2))
print(f"Bayes Analysis Height RMSE: {bayes_height_misfit_rmse}")
print(f"Bayes Analysis Total RMSE: {bayes_total_misfit_rmse}")

Bayes Analysis Height RMSE: 0.014782742194394686
Bayes Analysis Total RMSE: 0.008580741242779005


In [ ]:
# dci_analysis = run_assimilation(problem_params, solver_params, stations, y_obs, obs_per_window,
#                              obs_indices_per_window,
#                              H, B_inv, R_inv, P_inv, hb,
#                              'tidal',
#                              create_problem_solver,
#                              dci_cost_function, 
#                             grad_dci_cost_function
#                              )
# dci_rmse = np.sqrt(np.mean((true_signal.vals[:,:,0] - dci_analysis[:,:,0]) ** 2))
# dci_rmse

In [ ]:
# dci_wme_analysis = run_assimilation(problem_params, solver_params, stations, y_obs, obs_per_window,
#                              obs_indices_per_window,
#                              H, B_inv, R_inv, P_inv, hb,
#                              'tidal',
#                              create_problem_solver,
#                              dci_wme_cost_function,
#                              grad_dci_wme_cost_function
#                              )
# dci_wme_rmse = np.sqrt(np.mean((true_signal.vals[:,:,0] - dci_wme_analysis[:,:,0]) ** 2))
# dci_wme_rmse

In [ ]:
# plot_params = {
#     "lines.linewidth": 5,
#     "lines.markersize": 10,
#     'lines.markeredgecolor': 'black',
#     "legend.fontsize": 45,
#     "legend.frameon": False,
#     "xtick.labelsize": 45,
#     "ytick.labelsize": 45,
#     "axes.labelsize": 45,
#     "axes.labelpad": 10,
#     "axes.titlesize": 45,
#     "figure.figsize": (44, 16),
# }


# plot_simulation_results(true_signal, bayes_height_misfit_rmse, y_obs, hb, problem_params,
#                        obs_time_indices, plot_params, save=False, save_prefix="bayes_")

In [ ]:
# plot_params = {
#     "lines.linewidth": 5,
#     "lines.markersize": 10,
#     'lines.markeredgecolor': 'black',
#     "legend.fontsize": 45,
#     "legend.frameon": False,
#     "xtick.labelsize": 45,
#     "ytick.labelsize": 45,
#     "axes.labelsize": 45,
#     "axes.labelpad": 10,
#     "axes.titlesize": 45,
#     "figure.figsize": (44, 16),
# }


# plot_simulation_results(true_signal, bayes_analysis, y_obs, hb, problem_params,
#                        obs_indices, plot_params, save=False, save_prefix="bayes_")